# Business Case

## Project Overview

This project tackles the challenge of ensuring on-time deliveries and dependable operations within the air cargo industry by using passenger sentiment data as an indirect measure of overall airline performance.The goal is to work with airline partners to maintain high performance, reduce delays, and minimise disruptions that can impact the supply chain and customer satisfaction.


## Objectives and Scope

Objectives:

- Identify airlines known for their reliable performance (Khan et al., 2025).
- Leverage passengers' feedback and operational data to reduce delays and disruptions.
- Deliver practical insights to streamline logistics and improve supply chain processes.

Scope:

- Collect and analyse data from passenger experiences.
- Analyse issues such as delays as this also affect air cargo reliability.
- Develop recommendations to enhance partner collaboration with airlines.

Business Value:

- Protect Brand Reputation
- Optimise Airline Partner Selection
- Increase Operational Efficiency
- Increase Customer Loyalty
- Prevent Contractual Penalties


Data Sources Identification:

- To support these objectives, the project utlise web scraping techniques  (Khder, 2021) to extract passenger sentiment data from the Airline Quality Reviews website. This real-time data, combined with an ongoing machine learning project to predict flight delays (Tang, 2021), will form the basis for a comprehensive analysis of airline reliability and quality of service. Moreover, while direct feedback from cargo customers is limited, passenger reviews offer valuable insights into operational performance that directly impact both passenger satisfaction and cargo delivery reliability.

# Scraping

## Data Collection Process

Web scraping, or web data extraction, refers to the method of acquiring and organising data from the internet through automated techniques. It enables the extraction of large volumes of data from a wide range of online resources (Khder, 2021).

### Import Libraries

In [ ]:
%pip install --quiet BeautifulSoup4 requests
!pip install --upgrade numpy
!pip install --upgrade gensim
! pip install pyLDAvis

In [ ]:
import pandas as pd # data manipulation and storage
import requests  # To make HTTP requests and retrieve web page content
from bs4 import BeautifulSoup  # To extract data and parse HTML content
import time # To pause between requests
import os # To check file paths and file existence
from typing import List  # For type hints
from textblob import TextBlob  # For sentiment analysis
from wordcloud import WordCloud # to Visualise the results
import string  # Import the string module for punctuation
import re  # For module for regular expressions
import matplotlib.pyplot as plt # To create visualisations
import seaborn as sns # To create visualisations
import nltk # For natural language processing tasks
from nltk.stem import WordNetLemmatizer, PorterStemmer # For stemming and lemmatization
from nltk.corpus import stopwords # For removing stop words
from nltk.tokenize import TweetTokenizer # For tokenizing text
nltk.download('stopwords')
nltk.download('wordnet')
import numpy as np # For numerical operations
import gensim # For topic modeling
from gensim import corpora # For creating a corpus
import pyLDAvis # For interactive topic model visualisation
import pyLDAvis.gensim_models # For using pyLDAvis with gensim models

### Checking Site Rules



This function is really important as it checks the website's rules for what is allowed and disallowed in terms of web scraping.

For example: If Scrapy is used as the web crawler, it is permitted to scrape any part of the website without restrictions. However, if a different web crawler is used or if no specific rules are provided, the rules No 3, 4 and 5 must be followed.



 1. Scrapy: Allowed to access all parts of the website.

 2. TermlyBot: No specific rules, so it follows the general rules for all user-agents.

 3. All Web Crawlers:
  -Disallowed: /go/ directory, /wp-admin/ directory

 4.  Crawl Delay:
Wait at least 5 seconds between requests.

Sitemap:

 5. Use the sitemap at https://www.airlinequality.com/sitemap.xml to find important pages.

In [ ]:
# Check rules for web scraping
def check_robots() -> str:
  """
  Check rules for web scraping, ensuring compliance with ethical web scraping practices.

  Steps
  -------
  - Define a URL
  - Sends an HTTP GET request to the defined URL
  - Prints the HTTP status code and reason
  - Stores the response text
  - Prints the response

  Returns
  -------
  String with the list of what is allowed or disallowed

  Example Usage
  -------------
  check_robots()

  """
  url = "https://www.airlinequality.com/robots.txt"
  response = requests.get(url)
  print(f"{response.status_code} {response.reason}")
  text = response.text
  print(text)
  return text

check_robots()

### Airline Reviews Setup

This section is dedicated to setting up the process for extracting individual airline reviews from the downloaded HTML files. This setup is crucial as it defines the logic for identifying and extracting relevant data points (Airline Name | Title | Rating | Review_Text) from the HTML structure. The consistent formatting allows for scalable data extraction, required for larger datasets or additional airlines.


In [ ]:
# Scrape reviews from the extracted file
def scrape_reviews_from_file(file_path: str, max_reviews: int = 20) -> List[List[str]]:
    """
    Extracts airline reviews from a saved HTML file.
    The file is read locally, so no request is sent to the website at this step.

    Steps
    -------
    - Open the specified HTML file with UTF-8 encoding.
    - Parse the file's content using BeautifulSoup.
    - Finds all <article> tags with class 'comp_media-review-rated'.
    - For each review (up to 'max_reviews'):
        - Extracts the title from <h2> tag, or return "No Title" if missing.
        - Extracts the rating (the score before "/10" in the <div> with class "rating-10", or "No Rating").
        - Extract the review text from the <div> with class "text_content" (or "No Review").
    - Append the extracted data into a list.

    Parameters
    ----------
    file_path : The path to the HTML file.
    max_reviews : Maximum number of reviews to extract.

    Returns
    -------
    A list of reviews, where each review is a list: [Airline Name, title, rating, review_text].

    Example Usage
    -------------
    reviews = scrape_reviews_from_file(file_path='british-airways.html', max_reviews=20)
    """
    reviews = []

    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")
        articles = soup.find_all("article", class_="comp_media-review-rated")

        for article in articles[:max_reviews]:

            title_element = article.find("h2")
            title = title_element.text if title_element else "No Title"

            rating_element = article.find("div", class_="rating-10")
            rating = rating_element.text.strip().split("/")[0].strip() if rating_element and rating_element.text.strip() else "No Rating"

            review_text_element = article.find("div", class_="text_content")
            review_text = review_text_element.text if review_text_element else "No Review"

            reviews.append([title, rating, review_text])
    return reviews

In [ ]:
## List of specific airlines
airlines = [ "British Airways", "Lufthansa", "Emirates", "Qatar", "Singapore Airlines",
            "American Airlines" ]

### Making a GET request

In [ ]:
# Defining the url of the site
base_url = "https://www.airlinequality.com/airline-reviews/{airline}/"

## Data Transformation

Data cleaning and transformation are essential steps in data scraping, removing unwanted elements and formatting issues, and converting raw HTML data into a structured format for analysis and processing (Fraňo, 2024).

### Download HTML Files for Each Airline

This section is dedicated to downloading the HTML content of individual airline review pages and saving them locally. This step is crucial because storing HTML files locally ensures offline analysis and prevents server strain or IP blocking. Additionally, having static files ensures consistent analysis despite content changes.

Scalability Consideration

Currently, pages are downloaded one at a time with delays to comply with ethical scraping practices. For larger-scale scraping (WebScraping.AI, n.d.), this step could be parallelised for speed and automated using a workflow orchestration tool such as Prefect to enable regular updates across many airline pages (Prefect, n.d.).



In [ ]:
# Download and save HTML files for individual airline review pages
def download_airline_reviews() -> None:
    """
    Downloads the HTML content of individual airline review pages and saves them locally.

    Steps
    -------
    - Format Airline Name: Converts the airline name to lowercase and replaces spaces with hyphens.
    - Construct URL: Inserts the formatted airline name into the base URL.
    - Send HTTP Requesst: Sends a GET request to download the HTML content.
    - Check response:
      -  If request is successful (code == 200) → Saves the HTML content named after the formatted airline.
      -  If not successful → Prints error message and the status code.
    - Wait 5 seconds before the next request, as the site's robots.txt asks (crawl delay).

    This refined approach targets individual review pages rather than extracting reviews from a single aggregated page,
    ensuring accurate data collection for each airline.

    Returns
    -------
    None

    Example Usage
    -------------
    download_airline_reviews()
    """
    for airline in airlines:
        formatted_airline = airline.lower().replace(" ", "-")
        url = base_url.replace("{airline}", formatted_airline)

        response = requests.get(url)
        if response.status_code == 200:
            print(f"Downloading HTML for {airline}")

            with open(f"{formatted_airline}.html", "w", encoding="utf-8") as file:
                file.write(response.text)
            print(f"HTML file saved for {airline}")
        else:
            print(f"Failed to retrieve HTML for {airline} (Status code: {response.status_code})")

        # Crawl delay: wait 5 seconds between requests to the website, as robots.txt asks
        time.sleep(5)

### Process and Consolidate Reviews from Downloaded Files

This section is dedicated to processing and consolidating individual airline reviews extracted from downloaded HTML files, ensuring accurate extraction, organisation, and labeling for subsequent analysis.


In [ ]:
# Processes airline reviews from local HTML files
def process_airline_reviews(airline: str) -> List[List[str]]:
    """
    Processes and extracts reviews for a single airline.

     Steps
      -------
      - Format Airline Name: Converts the airline name to lowercase and replaces spaces with hyphens
      - File Path Construction: Construct the file path based on the formatted name
      - Check File Existence:
          - If file exists → Calls 'scrape_reviews_from_file' to extract reviews
          - If not exists → Prints error message
      - Label and append each extracted review with the airline name


    Returns:
    --------
    It returns a list of reviews for that specific airline.

    Example Usage
    -------------
    airlines_reviews = process_airline_reviews(airline='British Airways')
    """
    formatted_airline = airline.lower().replace(" ", "-")

    file_path = f"{formatted_airline}.html"

    airline_reviews = []

    if os.path.exists(file_path):
        print(f"Extracting reviews for {airline}")

        extracted_reviews = scrape_reviews_from_file(file_path)

        for review in extracted_reviews:
            airline_reviews.append([airline] + review)
        print(f"Found reviews for {airline}: {len(extracted_reviews)}")
    else:
        print(f"File not found for {airline}")

    return airline_reviews

In [ ]:
# Consolidates reviews from all airlines into a single list
def consolidate_reviews() -> List[List[str]]:
    """
    Consolidates into a list all the individual reviews from airlines.

    Steps
    -------
    - Iterate Airlines: Loops through the list of airlines.
    - Process Reviews: Calls the 'process_airline_reviews' function for each airline to extract reviews.
    - Extend Reviews: Adds the extracted reviews to a main list.

    Returns
    -------
    A consolidated list of all reviews for every airline.

    Example Usage
    -------------
    reviews = consolidate_reviews()
    """
    all_reviews = []
    for airline in airlines:
        reviews = process_airline_reviews(airline)
        all_reviews.extend(reviews)
    return all_reviews

## Data Storage

This section is dedicated to modifying the existing downloaded files to a more suitable format for analysis and also saving the extracted airline reviews to a local file. This step is essential for ensuring the data is kept safe and backed up, as well as facilitating the analysis of airline partners. It helps prevent data loss and supports an efficient analysis workflow.


### Save All Reviews to a File

In [ ]:
# Save reviews to a file
def save_reviews_to_file(reviews: List[List[str]], filename: str = "all_reviews.txt") -> None:
  """
    Saves a list of airline reviews to a file, allowing the user to scrape and analyse reviews offline.
    Uses 'with' statement for proper file clousure and resource management.

    Steps
    -------
    - File Opening: Opens the 'all_reviews.txt'.
    - Header Writing: Creater a header for columns "Airline Name | Title | Rating | Review Text"
    - Data Writing:Iterates over the list of reviews and writes each review as a row, separating values with a pipe.
    - Loops through each review.
    - Extract Data:
        - If the review has the expected format (4 elements):
            - Extracts airline, title, rating, and review text.
            - Split the review text column to separate the review and the review verification.
            - If successful, assigns the verification status; otherwise, marks as "Not Verified".
        - If the review format is unexpected, prints a message and skips it.
    - Write the file on the expected format

    Parameters
    ----------
    reviews : A list where each review is represented as a list of strings.
    filename : The name of the file where all reviews will be saved.

    Returns
    -------
    None

    Example Usage
    -------------
    save_reviews_to_file(reviews= reviews, filename= 'all_reviews.txt')
    """

  with open(filename, "w", encoding="utf-8") as file:
    file.write("Airline Name | Title | Rating | Verified | Review_Text\n")
    file.write("\n")

    for review in reviews:
        if len(review) == 4:
            airline, title, rating, review_text = review

            split_review = review_text.split("|", 1)

            if len(split_review) == 2:
                verified, review_text = split_review
                verified = verified.strip()
                review_text = review_text.strip()
            else:
                verified = "Not Verified"

        else:
            print(f"Skipping review with unexpected format: {review}")
            continue

        file.write(f"{airline} | {title} | {rating} | {verified} | {review_text}\n")

### Read and Print the Saved Reviews

This section is dedicated to verify that data was saved to a file sucessfully, and also provide a preview of the extracted reviews before applying NLP techniques.


In [ ]:
# Reads the saved reviews file and prints the reviews
def read_and_print_reviews(filename: str = "all_reviews.txt"):
  """
    Reads and prints the contents of a saved airline reviews file.
    Verifies data storage and provides a quick review of extracted reviews.

    Steps
    -------
    - Open File: Opens 'all_reviews.txt' file in read mode.
    - Read Content: Reads the content.
    - Print: Displays the content.

    Parameters
    ----------
    filename : The name of the file to read

    Returns
    -------
    None

    Example Usage
    -------------
    read_and_print_reviews(filename= 'all_reviews.txt')
    """
  with open(filename, "r", encoding="utf-8") as file:
      content = file.read()
      print(content)

# Natual Language Processing

In today's world, 80% of all data is unstructured (Eberendu, 2016), and Natural Language Processing (NLP) is key to unlocking its hidden value (Ravikumar et al., 2023). NLP transforms raw text into structured insights and has everyday applications such as chatbots, language translation, and speech recognition, facilitating many daily tasks. Despite challenges like context interpretation and language evolution, NLP continues to advance, enabling informed decision-making.

## Loading the File


This section is dedicated to introduce the 'load_reviews_df' function, which is tasked to load the airline reviews data. This step is important because even if the data is present in the notebook, including this function reinforces the principles of modularity, resulting in more organised, maintainable, reusable, and testable code, especially in larger projects or when using workflow orchestration tools.

In [ ]:
# Load the reviews data
def load_reviews_df(file_path: str = "all_reviews.txt") -> pd.DataFrame:
    """
    Loads reviews from a text file.

    Steps
    -------
    - Open the txt file originated from the scraping code,with the specific format "|" as the delimiter.
    - Skip the header.
    - Strip spaces from the columns.

    Parameters
    ----------
    file_path : The path to the reviews text file with all reviews


    Returns
    -------
    A DataFrame containing the reviews data


    Example Usage
    -------------
    data = load_reviews_df(file_path= 'all_reviews.txt')
    """
    df = pd.read_csv(file_path, sep="|", encoding="utf-8", skiprows=[1])
    df.columns = df.columns.str.strip()

    # Strip the spaces left around each value by the " | " separator (e.g. " Not Verified "),
    # so values match exactly in the verification analysis and in chart labels
    for column in df.select_dtypes(include=["object", "string"]).columns:
        df[column] = df[column].str.strip()

    return df

## Text Cleaning Functions

Text cleaning plays a crucial role in Natural Language Processing as it improves the accuracy of analysis by eliminating unnecessary words or components within a sentence, including stop words, punctuation, URLs, numbers, and very short words that do not to add meaningful context to passengers' sentiments towards the airline. By removing these elements, the analysis can focus only on the relevant content, improving the accuracy and reliability of the insights generated (Tabassum and Patil, 2020).

The following functions perform text cleaning iteratively, meaning each step builds upon the previous one to refine the text data. This process ensures the text is cleaned and ready for analysis.

In [ ]:
# Remove Stop Words
STOPWORDS = set(stopwords.words('english'))
def cleaning_stopwords(text: str) -> str:
    """Remove common stopwords from the text"""
    return " ".join([word for word in str(text).split() if word.lower() not in STOPWORDS])

In [ ]:
# Remove Punctuation
def cleaning_punctuations(text: str) -> str:
    """Remove punctuation characters from the text, including curly quotes such as ’ and “ ”"""
    return text.translate(str.maketrans('', '', string.punctuation + "’‘“”"))

In [ ]:
# Remove Repeating Characters
def cleaning_repeating_chars(text: str) -> str:
    """Reduce characters repeated 3+ times (e.g., 'soooo' becomes 'so') while keeping normal double letters ('food', 'class')."""
    return re.sub(r'(.)\1{2,}', r'\1', text)

In [ ]:
# Remove URLs
def cleaning_urls(text: str) -> str:
    """Remove URLs from the text"""
    return re.sub(r'https?://\S+|www\.\S+', '', text)

In [ ]:
# Remove Numbers
def cleaning_numbers(text: str) -> str:
    """Remove numbers from the text"""
    return re.sub(r'\d+', '', text)

In [ ]:
# Remove Short words
def remove_short_words(text: str) -> str:
    """Remove words that are 2 characters or fewer"""
    return " ".join(word for word in text.split() if len(word) > 2)

## Text Preprocessing

Preprocessing is another crucial step in Natural Language Processing (NLP) as it transforms raw, unstructured text into a cleaner and more meaningful format that models can understand and learn from. Without proper preprocessing, inconsistencies in the data can lead to inaccurate or misleading results.

1. Tokenization is important because it breaks down text into individual words, allowing NLP algorithms to process and analyse the structure of language. For example, "The flight was delayed" becomes ["The", "flight", "was", "delayed"] (Tabassum and Patil, 2020).

2. Stemming helps by reducing words to their base form, which consolidates similar terms and reduces the dimensionality of the dataset. For example, "delayed", "delaying", and "delay" all become "delay". By grouping these words together, a more accurate overall picture of how frequently the word is mentioned, making it easier to identify the issue (Tabassum and Patil, 2020).

3. Lemmatization enhances linguistic accuracy by converting words to their correct base form based on context, improving the quality of analysis and interpretation. For example, "was" -> "be" and "running" -> "run". By grouping different words belonging to the same concept, the result of the word count is more accurate (Tabassum and Patil, 2020).

In [ ]:
# Tokenize Text
def tokenize_text(text: str) -> list:
    """Tokenize the text using TweetTokenizer"""
    tokenizer = TweetTokenizer()
    return tokenizer.tokenize(text)

In [ ]:
# Apply Stemming
def apply_stemming(tokens: list) -> list:
    """Apply stemming to a list of tokens"""
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]

In [ ]:
# Apply Lemmatization
def apply_lemmatization(tokens: list) -> list:
    """Apply lemmatization to a list of tokens"""
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

## Combined Functions

This section is dedicated to introducing two functions to streamline text processing and ensure standardised text for analysis, leading to more accurate and reliable results. By encapsulating these steps into one pipeline, the process is organised and the code runs in the correct order.


In [ ]:
# Apply each step of text preprocessing
def process_tokens(text: str) -> str:
    """
    Tokenizes the text, then applies stemming and lemmatization
    Returns the processed text as a single string

    Steps
    -----
    - Tokenize the input text using the 'tokenize_text' function.
    - Apply stemming to the tokens using the 'apply_stemming' function.
    - Apply lemmatization to the tokens using the 'apply_lemmatization' function.
    - Join the processed tokens into a single string and return it.

    Parameters
    ----------
    The text to be processed

    Returns
    -------
    The text as a single string with tokens that have been stemmed and lemmatized
    """

    tokens = tokenize_text(text)
    tokens = apply_stemming(tokens)
    tokens = apply_lemmatization(tokens)
    return " ".join(tokens)

In [ ]:
# Apply each step of text cleaning, including preprocessing
def full_cleaning(text: str) -> str:
    """
    Applies a sequence of cleaning steps to the text

    Steps
    -----
      - Remove stopwords.
      - Remove punctuation.
      - Remove repeating characters.
      - Remove URLs.
      - Remove numbers.
      - Remove very short words.
      - Tokenize, then apply stemming and lemmatization.
    Returns the fully cleaned text.

    Parameters
    ----------
    The  text to be fully cleaned

    Returns
    -------
    The cleaned text
    """
    text = cleaning_stopwords(text)
    text = cleaning_punctuations(text)
    text = cleaning_repeating_chars(text)
    text = cleaning_urls(text)
    text = cleaning_numbers(text)
    text = remove_short_words(text)
    text = process_tokens(text)
    return text

## Sentiment Analysis

This project aims to analyse passenger sentiment regarding their experiences with various airlines. By identifying which partners are more likely to experience delays, the company can pinpoint areas of weakness. Understanding these weaknesses will enable the company to collaborate with these partners to reduce operational issues and strengthen relationships.

To help understanding this sentiment, sentiment analysis can be applied with polarity - a technique used to identify if the text is positive, negative or neutral (from -1 to 1, where -1 is negative sentiment, 0 neutral, and 1 positive sentiment.) (Yaqub et al., 2018).

On the other hand, subjectivity is about how much the text shows someone's personal thoughts or feelings by sharing their opinion on the subject. from 0 to 1 where 0 is objective (basded on facts) and 1 subjective (based on opnions) (Yaqub et al., 2018).

For example: on the first sentence (check in and security at T5 quick. Both Galle...), has a polarity scores of 0.21, meaning is slightly positive sentiment towards the airline and indicating that this text contains more opnions than facts (Yaqub et al., 2018).

In [ ]:
# Get Subjectivity Score
def get_subjectivity(text: str) -> float:
    """
    Returns the subjectivity score of the text using TextBlob
    These are the personal opinions, feelings, and beliefs of the
    individual who created the text.
    """
    return TextBlob(text).sentiment.subjectivity

In [ ]:
# Get Polarity Score
def get_polarity(text: str) -> float:
    """
    Returns the polarity score of the text using TextBlob
    This is a measure of how positive or negative the text is
    """
    return TextBlob(text).sentiment.polarity

In [ ]:
# Get Sentiment Label
def get_sentiment_label(score: float) -> str:
    """Classifies sentiment based on polarity score"""
    if score < 0:
        return 'Negative'
    elif score == 0:
        return 'Neutral'
    else:
        return 'Positive'

In [ ]:
# Calculate and Display Sentiment Percentages
def calculate_sentiment_percentage(df_positive, df_negative, df_neutral, data):
    """Calculate and display the percentage of positive, negative, and neutral reviews."""
    positive_percentage = round((df_positive.shape[0] / data.shape[0]) * 100, 1)
    negative_percentage = round((df_negative.shape[0] / data.shape[0]) * 100, 1)
    neutral_percentage = round((df_neutral.shape[0] / data.shape[0]) * 100, 1)

    print(f"Positive Reviews: {positive_percentage}%")
    print(f"Negative Reviews: {negative_percentage}%")
    print(f"Neutral Reviews: {neutral_percentage}%")

## NLP Pipeline

This section is dedicated to introducing the nlp_pipeline function, which integrates:
1. Full_Cleaning (preprocessing and cleaning steps)
2. Sentiment Analysis  (Subjectivity and Polarity)
3. Sentiment Labels

This approach ensures consistent data processing in order to extract insights from the data.

In [ ]:
# NLP pipeline to airline reviews
def nlp_pipeline(data: pd.DataFrame, review_column: str = "Review_Text") -> pd.DataFrame:
    """
    This function ensure the whole pipeline is applied to the reviews

    Steps
    -------
    - Calls the full_cleaning function to clean the review text, tokenizes and processes tokens (stemming and lemmatization).
    - Calls the get_subjectivity function to compute the subjectivity of the text.
    - Calls the get_polarity function to compute the polarity of the text.
    - Classifies the sentiment.

    Parameters
    ----------
    The input DataFrame containing review data

    Returns
    -------
    The original DataFrame with new columns

    Example Usage
    -------------
    processed_data = nlp_pipeline(data= data, review_column='Review_Text')

    """

    data["Clean_Review"] = data[review_column].apply(full_cleaning)
    # Sentiment is scored on the original text: TextBlob needs words such as "not" and full word forms,
    # which cleaning removes. The cleaned text is used for word clouds and topic modelling.
    data["Subjectivity"] = data[review_column].apply(get_subjectivity)
    data["Polarity"] = data[review_column].apply(get_polarity)
    data["Sentiment"] = data["Polarity"].apply(get_sentiment_label)

    return data

## Data Analysis

This section focuses on analysing passenger experiences and operational challenges reflected in airline reviews to gain insights relevant to air cargo operations. By quantifying the credibility of reviews through verification status analysis, the project ensures data reliability. Furthermore, by identifying and quantifying mentions of delays and cancellations within the reviews, the project pinpoints specific areas of potential underperformance, supporting data-driven decisions about airline partnerships and strategies for improving air cargo delivery reliability.


In [ ]:
# Analyse Verification Status
def analyse_verification_status(data):
    """
    Analyses the verification status of reviews and prints the distribution.

    Steps
    -----
    - Calculate the total number of reviews.
    - Count the number of verified and non-verified reviews.
    - Calculate the percentage of verified and non-verified reviews.
    - Print the verification status.
    - Check non-verified reviews among negative sentiment

    Parameters
    ----------
    The DataFrame containing the reviews

    Returns
    -------
    None

    Example Usage
    -------------
    analyse_verification_status(processed_data)
    """
    total_reviews = len(data)
    verified_reviews = data['Verified'].value_counts().get('✅ Trip Verified', 0)
    non_verified_reviews = data['Verified'].value_counts().get('Not Verified', 0)
    verified_percentage = round((verified_reviews / total_reviews) * 100, 1)
    non_verified_percentage = round((non_verified_reviews / total_reviews) * 100, 1)

    print(f"\nVerification Status Distribution:")
    print(f"Total Reviews: {total_reviews}")
    print(f"Verified Reviews: {verified_reviews} ({verified_percentage}%)")
    print(f"Non-Verified Reviews: {non_verified_reviews} ({non_verified_percentage}%)")

    # Negative sentiment analysis (Non-Verified)
    negative_reviews = data[data['Sentiment'] == 'Negative']
    negative_non_verified = negative_reviews[negative_reviews['Verified'] == 'Not Verified']
    negative_non_verified_percentage = round((len(negative_non_verified) / len(negative_reviews)) * 100, 1) if len(negative_reviews) > 0 else 0

    print(f"\nNon-Verified Reviews Among Negative Sentiment:")
    print(f"Total Negative Reviews: {len(negative_reviews)}")
    print(f"Non-Verified Negative Reviews: {len(negative_non_verified)} ({negative_non_verified_percentage}%)")

In [ ]:
# Analyse Delays
def analyse_delays(data: pd.DataFrame) -> None:
    """
    Analyses delays and cancellations within the review data,
    focusing on negative sentiment and counting delay keywords per airline.

    Steps
    -----
        - Define a list of keywords.
        - Filter the DataFrame to include only reviews with negative sentiment.
        - Create a dictionary to store the count of delay keywords for each airline.
        - Iterate through the negative reviews and count the delay keywords for each airline.
        - Update the dictionary with the keyword counts.
        - Convert the dictionary into a DataFrame with airlines and their delay keyword counts.

    Parameters
    ----------
     The DataFrame containing processed reviews.

    Returns
    -------
    A DataFrame with airlines and their delay keyword counts.

    Example Usage
    -------------
    analyse_data(data= data)
    """
    delay_keywords = ["delay", "delays", "delayed", "delaying", "late",
                      "cancel", "cancels", "cancelled", "canceled", "cancellation", "cancellations"]
    df_negative = data[data['Sentiment'] == 'Negative']

    airline_keyword_counts = {}

    for index, row in df_negative.iterrows():
        airline = row['Airline Name']
        # Use the original text: stemming turns 'cancelled' into 'cancel', so it would never match the keyword list.
        review_text = cleaning_punctuations(str(row['Review_Text']).lower())

        keyword_count = sum(1 for keyword in delay_keywords if keyword in review_text.split())

        airline_keyword_counts.setdefault(airline, 0)
        airline_keyword_counts[airline] += keyword_count

    result_df = pd.DataFrame(list(airline_keyword_counts.items()), columns=['Airline Name', 'Delay_Keyword_Count'])

    return result_df

## Visualisations

Making sense of large amounts of data presents a significant challenge. To address this, visualisations play a crucial role by identifying underlying patterns and the extraction of valuable insights. Furthermore, these visual representations offer a quick and effective method for communicating complex results in a way that is easily understood by anyone, regardless of their technical expertise.

In [ ]:
# Create DataFrames for Positive, Negative, and Neutral Reviews
def create_sentiment_dataframes(data):
    """Create separate DataFrames for positive, negative, and neutral reviews."""
    df_positive = data[data['Sentiment'] == 'Positive']
    df_negative = data[data['Sentiment'] == 'Negative']
    df_neutral = data[data['Sentiment'] == 'Neutral']
    return df_positive, df_negative, df_neutral

In [ ]:
# Generate Word Clouds for Sentiment
def generate_word_clouds(df_positive, df_negative, df_neutral):
    """Generate and display word clouds for positive, negative, and neutral reviews."""
    for label, df_group in [("Positive", df_positive), ("Negative", df_negative), ("Neutral", df_neutral)]:
        text = ' '.join(df_group['Clean_Review'])

        # Skip a group with no reviews: WordCloud raises an error on empty text
        if not text.strip():
            print(f"No {label.lower()} reviews - word cloud skipped.")
            continue

        wordcloud = WordCloud(width=800, height=500, random_state=42, max_font_size=100).generate(text)

        plt.figure(figsize=(15, 8))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f"{label} Reviews Word Cloud")
        plt.show()

In [ ]:
# Plot Polarity vs Subjectivity
def plot_polarity_subjectivity(data):
    """Plot polarity vs subjectivity of the reviews."""
    plt.figure(figsize=(8,6))
    for i in range(0, data.shape[0]):
        plt.scatter(data['Polarity'][i], data['Subjectivity'][i], color='Blue')

    plt.title('Sentiment Analysis (Polarity vs Subjectivity)')
    plt.xlabel('Polarity')
    plt.ylabel('Subjectivity')
    plt.show()

In [ ]:
# Plot Sentiment Counts
def plot_sentiment_counts(data):
    """Create a bar chart to visualise the sentiment distribution."""
    plt.title('Sentiment Analysis')
    plt.xlabel('Sentiment')
    plt.ylabel('Count')
    data['Sentiment'].value_counts().plot(kind='bar')
    plt.show()

In [ ]:
# Sentiment vs. Airline
def plot_sentiment_by_airline(data: pd.DataFrame):
    """Creates a boxplot of review polarity for each airline."""
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Airline Name', y='Polarity', data=data)
    plt.title('Sentiment Distribution by Airline')
    plt.xlabel('Airline')
    plt.ylabel('Polarity')
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
# Calculate average rating for each airline (all reviews)
def plot_average_rating_by_airline(data: pd.DataFrame):
    """
    Computes average ratings overall and for negative reviews, then displays a grouped bar chart.
    """
    data = data.copy()
    data['Rating'] = pd.to_numeric(data['Rating'], errors='coerce')

    overall_ratings = data.groupby('Airline Name')['Rating'].mean().reset_index()
    overall_ratings['Rating_Type'] = 'Overall Average'

    negative_reviews = data[data['Sentiment'] == 'Negative']
    negative_ratings = negative_reviews.groupby('Airline Name')['Rating'].mean().reset_index()
    negative_ratings['Rating_Type'] = 'Negative Average'

    all_ratings = pd.concat([overall_ratings, negative_ratings])

    plt.figure(figsize=(12, 6))
    sns.barplot(x='Airline Name', y='Rating', hue='Rating_Type', data=all_ratings, palette={'Overall Average': 'blue', 'Negative Average': 'slategrey'})  # Specify palette
    plt.title('Average Rating vs. Negative Average Rating by Airline')
    plt.xlabel('Airline')
    plt.ylabel('Average Rating')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Rating Type')
    plt.show()

In [ ]:
# Create a bar plot
def plot_delay_keywords(delay_keywords_df: pd.DataFrame):
    """
    Plots a bar chart showing delay- and cancellation-related keyword counts by airline.
    """
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Airline Name', y='Delay_Keyword_Count', data=delay_keywords_df, color='blue')
    plt.title('Frequency of Delay-Related Keywords by Airline')
    plt.xlabel('Airline')
    plt.ylabel('Count of Delay Keywords')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## Topic Modelling

This section aims to identify common words or sentences and group them in topics (Murel and Kavlakoglu, 2024). This method helps go beyond sentiment analysis to understand specific reasons behind passenger feedback. By understand the topics of discussion among passengers, can lead to actinable insights to improve operations and ensure on-time cargo deliveries.

In [ ]:
# Topic Modeling with LDA
def preprocess_for_topic_modelling(data, review_column="Clean_Review"):
    """
   Prepares text for topic modeling.

   Steps
    -----
    - Get the cleaned reviews from the specified column in the DataFrame.
    - Convert the list of cleaned reviews (already tokenized) into a format suitable for topic modeling.
    - Create a dictionary and a corpus (Bag of Words) directly from the processed reviews.

    Parameters
    ----------
    The DataFrame containing the reviews

    Returns
    -------
    A tuple containing the dictionary and corpus for topic modeling

    Example Usage
    -------------
    preprocess_for_topic_modelling(data= data, review_column='Clean_Review')
    """
    processed_reviews = data[review_column].tolist()

    processed_reviews = [review.split() for review in processed_reviews]

    dictionary = corpora.Dictionary(processed_reviews)
    corpus = [dictionary.doc2bow(review) for review in processed_reviews]

    return dictionary, corpus

In [ ]:
# Apply LDA Model
def apply_lda_model(corpus, dictionary, num_topics=5):
    """
    Apply LDA model for topic modeling using the given corpus and dictionary.

    Steps
    -----
    - Set the number of topics to find.
    - Initialise the LDA model with the given corpus, dictionary, and number of topics.
    - Train the model with multiple passes over the corpus.
    - Display the top words for each identified topic.

    Parameters
    ----------
    The corpus in Bag of Words
    The dictionary mapping word IDs to words
    The number of topics to identify, by default 5

    Returns
    -------
    The trained LDA model

    Example Usage
    -------------
    lda_model = apply_lda_model(corpus, dictionary, num_topics=5)
    """

    lda_model = gensim.models.LdaMulticore(corpus, num_topics=num_topics,
                                          id2word=dictionary, passes=10, random_state=42,
                                          workers=2)


    topics = lda_model.show_topics(num_topics=num_topics, num_words=5, formatted=False)

    print("Identified Topics:\n")
    for topic_id, topic in topics:
        print(f"Topic #{topic_id + 1}: {', '.join([word for word, prob in topic])}")

    return lda_model

In [ ]:
# Visualisation of LDA Topics
def visualise_lda_topics(lda_model, corpus, dictionary):
    """
    Visualise the topics generated by LDA
    """
    pyLDAvis.enable_notebook()
    vis_data = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary, mds='mmds')
    display(vis_data)
    plt.show()

# Main functions

This section is dedicated to structuring the project around main functions, each handling a specific step in the data workflow. This modular approach improves organisation, enhances reusability, simplifies maintenance, increases readability, and facilitates collaboration. By using main functions, the project becomes more modular, maintainable, and scalable, boosting overall efficiency and effectiveness (Davis, Burry & Burry, 2011).

In [ ]:
def main_nlp():
    """
    Loads the airline review data, processes it through the NLP pipeline,
    and performs various analyses.
    """
    # Load the data
    data = load_reviews_df()

    # Process the data through the NLP pipeline
    processed_data = nlp_pipeline(data, review_column="Review_Text")

    # Perform verification analysis
    analyse_verification_status(processed_data)

    # Create sentiment dataframes
    df_positive, df_negative, df_neutral = create_sentiment_dataframes(processed_data)

    # Generate and display word clouds
    generate_word_clouds(df_positive, df_negative, df_neutral)

    # Plot polarity vs subjectivity
    plot_polarity_subjectivity(processed_data)

    # Calculate and display sentiment percentages
    calculate_sentiment_percentage(df_positive, df_negative, df_neutral, processed_data)

    # Plot sentiment counts
    plot_sentiment_counts(processed_data)

    plot_sentiment_by_airline(processed_data)

    plot_average_rating_by_airline(processed_data)

    # Analyse Delays by airlines
    delay_keywords_df = analyse_delays(df_negative)
    print(delay_keywords_df)
    plot_delay_keywords(delay_keywords_df)

    # Preprocess for topic modeling
    dictionary, corpus = preprocess_for_topic_modelling(processed_data)

    # Apply LDA model
    lda_model = apply_lda_model(corpus, dictionary)

    # Visualize LDA topics
    visualise_lda_topics(lda_model, corpus, dictionary)

In [ ]:
# Main function to run the whole process.
def main():
    """
    Runs the complete process for scraping and handling airline reviews.

    Steps
    -------
    - Check Robots.txt
    - Download Airline Reviews
    - Consolidate Reviews
    - Save Reviews
    - Read and Print Reviews
    - Run NLP Analysis

    Returns
    -------
    None
    """
    check_robots()
    download_airline_reviews()
    reviews = consolidate_reviews()
    save_reviews_to_file(reviews)
    read_and_print_reviews()
    main_nlp()


if __name__ == "__main__":
      main()

# References

1. Davis, D., Burry, J. and Burry, M., 2011. Understanding visual scripts: improving collaboration through modular programming. Available at: https://www.danieldavis.com/papers/ijac_davis_burry_burry.pdf [Accessed 26 April 2025].


2. Eberendu, A.C., 2016. Unstructured Data: an overview of the data of Big Data. <i>International Journal of Computer Trends and Technology</i>, 38(1), pp.46-50.Available at:https://www.researchgate.net/profile/Adanma-Eberendu/publication/309393428_Unstructured_Data_an_overview_of_the_data_of_Big_Data/links/5bc89b5c458515f7d9c65beb/Unstructured-Data-an-overview-of-the-data-of-Big-Data.pdf [Accessed 09 April 2025].

3. Fraňo, M., 2024. <i> Web Scraping as a Data Source for Machine Learning Models and the Importance of Preprocessing Web Scraped Data<i/> (Bachelor's thesis, University of Twente). Available at: https://essay.utwente.nl/100867/ [Accessed 06 April 2025].

4. Khan, M.H., Naseem, U., Keya, A.J., Maitra, S., Mithu, K., & Alam, M.G.R., 2025. Systematic Literature Review on Sentiment Analysis in Airline Industry. <i>SN Computer Science</i>. Available at: <https://link.springer.com/article/10.1007/s42979-024-03567-w> [Accessed 12 April 2025].

5. Khder, M.A., 2021. Web scraping or web crawling: State of art, techniques, approaches and application. <i>International Journal of Advances in Soft Computing & Its Applications</i>, 13(3). Available at: <http://ijasca.zuj.edu.jo/PapersUploaded/2021.3.11.pdf> [Accessed 09 April 2025].

6. Murel, J. and Kavlakoglu, E., 2024. <i>What is topic modeling?</i> *IBM*. [online] Available at: <https://www.ibm.com/think/topics/topic-modeling> [Accessed 23 April 2025].

7. Prefect. (n.d.). <i>Run tasks concurrently or in parallel. Retrieved April 25, 2025</i>. [online] Available at: https://docs.prefect.io/v3/develop/task-runners [Accessed 28 April 2025].

8. Ravikumar, V., Kodmelwar, M.K., Shekhawat, J., Sakhare, N.N. and Vijayalakshmi, V.J., 2023, December. Unlocking the Power of Natural Language Processing to Automate Knowledge Discovery. In <i> 2023 3rd International Conference on Smart Generation Computing, Communication and Networking (SMART GENCON)<i/> (pp. 1-6). IEEE. Availabe at : https://ieeexplore.ieee.org/abstract/document/10442434  [Accessed 16 April 2025].

9. Tabassum, A. and Patil, R.R., 2020. A survey on text pre-processing & feature extraction techniques in natural language processing. <i>International Research Journal of Engineering and Technology (IRJET)*, 7(06)</i>, pp.4864-4867. Available at:https://d1wqtxts1xzle7.cloudfront.net/64643112/IRJET-V7I6913-libre.pdf?1602332642=&response-content-disposition=inline%3B+filename%3DIRJET_A_Survey_on_Text_Pre_Processing_an.pdf&Expires=1745595406&Signature=Qb2yniUja-A~BoXUC9TAFMiCKDEAWf7Be4TAPYj9KsttumV-tQxitoEVdFjkZUVtIWNFUiDR~8ovmBPUWr27kZ3emBP0pnipDooQ55ZZq5aMTwJ7O53VAGHhsl55WulCq5ORW6fvtpEtHe~UqCFteWeJkwuDft8fWkh-f8xksesTght-JiWXwSXRmqQqnzFxzKb4oMhzi7awkOTW4GUqivJVbpDuo6WvGCACMEAC6~pqvnWo4Nm7Q3uKGYLOnK38d0Ya-1nKoRJbSsPd2pH92R-LUHnTFHq3gw1iIGRSFL9es9hEEbJytNqX4jny2Q52OFh333hxXDMvRCPeY12NYQ__&Key-Pair-Id=APKAJLOHF5GGSLRBV4ZA [Accessed 15 April 2025].

10. Tang, Y., 2021, October. Airline flight delay prediction using machine learning models. <i>In Proceedings of the 2021 5th International Conference on E-Business and Internet (pp. 151-154)</i>. Available at: <https://doi.org/10.1145/3497701.3497725> [Accessed 08 April 2025].

11.	WebScraping.AI, n.d. <i>What is the ideal time delay between requests to avoid throttling when scraping Homegate?</i> [online]. Available at: https://webscraping.ai/faq/homegate-scraping/what-is-the-ideal-time-delay-between-requests-to-avoid-throttling-when-scraping-homegate [Accessed: 22 April 2025].

12. Yaqub, U., Sharma, N., Pabreja, R., Chun, S.A., Atluri, V. and Vaidya, J., 2018, May. Analysis and visualization of subjectivity and polarity of Twitter location data. <i>In Proceedings of the 19th annual international conference on digital government research: governance in the data age</i> (pp. 1-10).Availabe at: https://dl.acm.org/doi/abs/10.1145/3209281.3209313 [Accessed 15 April 2025].



